# Sistem Pencetus Kandidat Neologisme Istilah Teknis (EN→ID)

Proyek riset NLP — Saga University / konversi CIF62340 FILKOM UB.

## Bagian 1 — Setup & Loading Model

In [ ]:
!pip install -q transformers sentencepiece sentence-transformers gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 28.0 MB/s eta 0:00:00


In [ ]:
import re
import numpy as np
from numpy import dot
from numpy.linalg import norm
from gensim.models import KeyedVectors
import torch
from sentence_transformers import SentenceTransformer

# Untuk memastikan hasil konsisten setiap kali notebook dijalankan ulang
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# Unduh dan ekstrak model FastText bahasa Indonesia (Common Crawl, 300 dimensi)
!wget -nc https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.id.300.vec.gz
!gunzip -k cc.id.300.vec.gz

FASTTEXT_PATH = "/content/cc.id.300.vec"
print("Memuat model FastText...")
ft_model = KeyedVectors.load_word2vec_format(FASTTEXT_PATH, limit=100000)
print(f"Model FastText berhasil dimuat. Jumlah kata: {len(ft_model.key_to_index)}")

--2026-06-19 08:31:02--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.id.300.vec.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.9.180.50, 65.9.180.116, 65.9.180.94, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.9.180.50|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1227018698 (1.1G) [binary/octet-stream]
Saving to: ‘cc.id.300.vec.gz’

cc.id.300.vec.gz    100%[===================>]   1.14G  32.3MB/s    in 37s     

2026-06-19 08:31:40 (31.3 MB/s) - ‘cc.id.300.vec.gz’ saved [1227018698/1227018698]

Memuat model FastText...
Model FastText berhasil dimuat. Jumlah kata: 100000


In [ ]:
# Model Sentence-Transformer multilingual -- dipilih setelah eksplorasi awal
# memakai IndoBERT base menunjukkan masalah anisotropy (lihat Lampiran di
# akhir notebook untuk catatan eksplorasi tersebut).
SBERT_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print("Memuat model Sentence-Transformer multilingual...")
sbert_model = SentenceTransformer(SBERT_MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sbert_model = sbert_model.to(device)

print(f"Model berhasil dimuat. Berjalan di: {device}")

Memuat model Sentence-Transformer multilingual...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model berhasil dimuat. Berjalan di: cpu


## Bagian 2 — Fungsi Embedding & Similarity Dasar

In [ ]:
def cosine_sim(a, b):
    """Menghitung cosine similarity antara dua vektor."""
    return dot(a, b) / (norm(a) * norm(b))


def get_sbert_embedding(text):
    """
    Mengubah teks menjadi vektor menggunakan model Sentence-Transformer.
    Mendukung input Inggris maupun Indonesia karena model bersifat multilingual.
    """
    return sbert_model.encode(text, convert_to_numpy=True)


def score_candidate(candidate_word, full_definition):
    """
    Menghitung skor kecocokan makna antara satu kandidat kata Indonesia
    dengan SELURUH definisi (sebagai satu kalimat utuh), bukan kata kunci
    satu-satu. Versi ini dipilih untuk mengurangi -- meski belum sepenuhnya
    menghilangkan -- bias terhadap kemiripan leksikal/substring yang
    ditemukan pada versi awal (lihat Bagian 7 dan Lampiran untuk diskusi).
    """
    emb_candidate = get_sbert_embedding(candidate_word)
    emb_definition = get_sbert_embedding(full_definition)
    return cosine_sim(emb_candidate, emb_definition)

## Bagian 3 — Data Acuan Proyek

Seluruh data tetap (definisi, kamus terjemahan, gold standard) dikumpulkan di satu tempat agar mudah ditelusuri dan diperluas.

In [ ]:
# Definisi istilah teknis berbahasa Inggris (ditulis manual, bukan disalin
# dari kamus berhak cipta)
definitions = {
    "array": "A collection of elements stored in sequential order.",
    "browser": "A software application used to access and navigate web pages.",
    "compiler": "A program that translates source code into machine code.",
    "mouse": "A pointing device used to control a cursor on a computer screen.",
    "string": "A sequence of characters used as data.",
}

In [ ]:
# Kamus terjemahan kata kunci (hasil ekstraksi definisi) EN -> ID.
# Cakupan terbatas, disesuaikan dengan kata kunci yang muncul di 5 istilah uji.
EN_ID_DICT = {
    # dari "array"
    "collection": "kumpulan",
    "elements": "elemen",
    "stored": "simpan",
    "sequential": "berurutan",
    "order": "urutan",

    # dari "browser"
    "software": "perangkat lunak",
    "application": "aplikasi",
    "access": "akses",
    "navigate": "navigasi",
    "web": "web",
    "pages": "halaman",

    # dari "compiler"
    "program": "program",
    "translates": "menerjemahkan",
    "source": "sumber",
    "code": "kode",
    "machine": "mesin",

    # dari "mouse"
    "pointing": "penunjuk",
    "device": "perangkat",
    "control": "kendali",
    "cursor": "kursor",
    "computer": "komputer",
    "screen": "layar",

    # dari "string"
    "sequence": "urutan",
    "characters": "karakter",
    "data": "data",
}

In [ ]:
# Padanan resmi/jawaban yang dianggap benar (gold standard) untuk setiap
# istilah, bersumber dari KBBI dan/atau situs PASTI (Badan Bahasa)
gold_standard = {
    "array": "larik",
    "browser": "peramban",
    "compiler": "pengompilasi",
    "mouse": "tetikus",
    "string": "untai",
}

## Bagian 4 — Preprocessing & Ekstraksi Kata Kunci

In [ ]:
# Daftar stopword bahasa Inggris yang umum (kata fungsi, bukan kata bermakna).
# Disusun manual dan ringkas untuk riset tahap awal -- bukan diambil dari
# library pihak ketiga (mis. NLTK) supaya tidak menambah dependensi.
ENGLISH_STOPWORDS = {
    "a", "an", "the", "of", "in", "on", "at", "to", "for", "with", "by",
    "is", "are", "was", "were", "be", "been", "being",
    "and", "or", "but", "if", "than", "that", "this", "these", "those",
    "used", "use", "uses", "using", "which", "who", "whom", "as", "it",
    "its", "from", "into", "such", "also", "can", "may", "often"
}


def extract_keywords(definition_text):
    """
    Mengekstrak kata kunci bermakna dari sebuah definisi Inggris,
    dengan membuang stopword dan tanda baca.
    """
    text_clean = re.sub(r"[^a-zA-Z\s]", "", definition_text.lower())
    words = text_clean.split()
    keywords = [w for w in words if w not in ENGLISH_STOPWORDS and len(w) >= 3]
    return keywords


def translate_keyword(keyword):
    """
    Menerjemahkan satu kata kunci Inggris ke Indonesia menggunakan kamus
    manual EN_ID_DICT. Mengembalikan None jika tidak ditemukan di kamus.
    """
    return EN_ID_DICT.get(keyword.lower())

In [ ]:
# Uji coba cepat
test_definition = "a sequence of characters used as data in a program"
print("Kata kunci yang diekstrak:", extract_keywords(test_definition))

test_keywords = ["sequence", "characters", "data", "program"]
translated = [translate_keyword(kw) for kw in test_keywords]
print("Hasil terjemahan:", list(zip(test_keywords, translated)))

Kata kunci yang diekstrak: ['sequence', 'characters', 'data', 'program']
Hasil terjemahan: [('sequence', 'urutan'), ('characters', 'karakter'), ('data', 'data'), ('program', 'program')]


## Bagian 5 — Pencarian Kandidat (FastText)

In [ ]:
def get_fasttext_candidates(indonesian_word, topn=10):
    """
    Mencari kata-kata Indonesia yang dekat secara vektor dengan
    sebuah kata Indonesia awal, menggunakan model FastText.
    Mengembalikan list kosong jika kata tidak ditemukan di vocabulary model.
    """
    if indonesian_word not in ft_model.key_to_index:
        return []
    similar_words = ft_model.most_similar(indonesian_word, topn=topn)
    return [word for word, score in similar_words]


def generate_candidates(term, definition, topn_per_keyword=10):
    """
    Pipeline untuk satu istilah: ekstrak kata kunci dari definisi,
    terjemahkan ke Indonesia, lalu cari kandidat FastText dari setiap
    kata kunci yang berhasil diterjemahkan. Sumber kandidat MURNI dari
    kata kunci definisi -- tidak ada jalur tambahan dari terjemahan
    harfiah istilah utama (jalur tersebut sudah dihapus karena terbukti
    menyebabkan evaluasi sirkular, lihat Lampiran C).
    """
    keywords_en = extract_keywords(definition)

    all_candidates = set()
    translated_seeds = []

    for kw in keywords_en:
        translated = translate_keyword(kw)
        if translated is None:
            continue
        translated_seeds.append(translated)
        all_candidates.update(get_fasttext_candidates(translated, topn=topn_per_keyword))

    all_candidates = {c for c in all_candidates if c.isalpha() and c.islower()}
    return list(all_candidates), translated_seeds

In [ ]:
# Uji coba cepat untuk istilah "string"
candidates, seeds = generate_candidates("string", definitions["string"])
print(f"Kata kunci yang diterjemahkan (seed): {seeds}")
print(f"Jumlah kandidat ditemukan: {len(candidates)}")
print(f"Contoh kandidat: {candidates[:15]}")

Kata kunci yang diterjemahkan (seed): ['urutan', 'karakter', 'data']
Jumlah kandidat ditemukan: 19
Contoh kandidat: ['datanya', 'menempati', 'statistik', 'urutannya', 'kepribadian', 'mendata', 'peringkat', 'karakternya', 'watak', 'urutkan', 'karakteristik', 'diurutkan', 'dokumen', 'database', 'berurutan']


## Bagian 6 — Ranking Kandidat (SBERT)

In [ ]:
def rank_candidates(term, definition, topn_per_keyword=10, top_results=10):
    """
    Pipeline lengkap untuk satu istilah: hasilkan kandidat dari FastText,
    lalu ranking berdasarkan skor kemiripan makna (SBERT) terhadap
    definisi penuh.
    """
    candidates, seeds = generate_candidates(term, definition, topn_per_keyword)

    scored_candidates = []
    for candidate in candidates:
        score = score_candidate(candidate, definition)
        scored_candidates.append((candidate, score))

    scored_candidates.sort(key=lambda x: x[1], reverse=True)
    return scored_candidates[:top_results]

In [ ]:
# Uji coba untuk istilah "string"
ranked = rank_candidates("string", definitions["string"])
print("Ranking kandidat untuk 'string':")
for i, (word, score) in enumerate(ranked, 1):
    print(f"{i}. {word} — skor: {score:.4f}")

Ranking kandidat untuk 'string':
1. datanya — skor: 0.6094
2. database — skor: 0.4708
3. berkarakter — skor: 0.3605
4. statistik — skor: 0.3604
5. karakternya — skor: 0.3294
6. dokumen — skor: 0.3271
7. urutkan — skor: 0.3247
8. tokoh — skor: 0.3050
9. karakteristik — skor: 0.3033
10. diurutkan — skor: 0.2999


## Bagian 7 — Evaluasi Kuantitatif (Recall/Precision/F1 @K)

### 7a. Recall@K (Exact Match)

Mengukur apakah jawaban resmi (gold standard) muncul **tepat** di antara top-K kandidat yang dihasilkan sistem. Ini kriteria yang ketat -- hanya dianggap "berhasil" jika kandidat sama persis dengan istilah resmi.

In [ ]:
def evaluate_recall_at_k(term, definition, gold_answer, k=5, topn_per_keyword=10):
    """
    Mengevaluasi apakah jawaban gold standard muncul di antara
    top-K kandidat yang dihasilkan sistem untuk satu istilah.
    """
    ranked = rank_candidates(term, definition, topn_per_keyword=topn_per_keyword, top_results=k)
    top_k_words = [word for word, score in ranked]

    hit = 1 if gold_answer in top_k_words else 0
    rank_position = top_k_words.index(gold_answer) + 1 if hit else None

    return {
        "term": term,
        "gold_answer": gold_answer,
        "top_k_candidates": top_k_words,
        "hit": hit,
        "rank_position": rank_position,
    }


def evaluate_all_terms(definitions_dict, gold_dict, k=5):
    """
    Menjalankan evaluasi untuk seluruh istilah, lalu menghitung
    metrik agregat: Recall@K, Precision@K, dan F1@K.
    """
    results = [
        evaluate_recall_at_k(term, definitions_dict[term], gold_dict[term], k=k)
        for term in gold_dict
    ]

    total = len(results)
    hits = sum(r["hit"] for r in results)

    recall_at_k = hits / total
    precision_at_k = hits / (total * k)
    if recall_at_k + precision_at_k > 0:
        f1_at_k = 2 * (precision_at_k * recall_at_k) / (precision_at_k + recall_at_k)
    else:
        f1_at_k = 0.0

    return {
        "individual_results": results,
        "recall_at_k": recall_at_k,
        "precision_at_k": precision_at_k,
        "f1_at_k": f1_at_k,
        "k": k,
    }

In [ ]:
eval_results = evaluate_all_terms(definitions, gold_standard, k=5)

print("=== Hasil Evaluasi per Istilah ===")
for r in eval_results["individual_results"]:
    status = f"DITEMUKAN di posisi {r['rank_position']}" if r["hit"] else "TIDAK DITEMUKAN"
    print(f"{r['term']:12s} -> gold: '{r['gold_answer']:12s}' | {status}")
    print(f"             Top-5 kandidat: {r['top_k_candidates']}")
    print()

print("=== Metrik Agregat ===")
print(f"Recall@{eval_results['k']}:    {eval_results['recall_at_k']:.4f}")
print(f"Precision@{eval_results['k']}: {eval_results['precision_at_k']:.4f}")
print(f"F1@{eval_results['k']}:        {eval_results['f1_at_k']:.4f}")

=== Hasil Evaluasi per Istilah ===
array        -> gold: 'larik       ' | TIDAK DITEMUKAN
             Top-5 kandidat: ['koleksi', 'elemennya', 'serangkaian', 'komponen', 'element']

browser      -> gold: 'peramban    ' | TIDAK DITEMUKAN
             Top-5 kandidat: ['web', 'webnya', 'software', 'aplikasinya', 'website']

compiler     -> gold: 'pengompilasi' | TIDAK DITEMUKAN
             Top-5 kandidat: ['kodenya', 'pengkodean', 'koding', 'code', 'programnya']

mouse        -> gold: 'tetikus     ' | TIDAK DITEMUKAN
             Top-5 kandidat: ['cursor', 'dikomputer', 'monitor', 'komputernya', 'layarnya']

string       -> gold: 'untai       ' | TIDAK DITEMUKAN
             Top-5 kandidat: ['datanya', 'database', 'berkarakter', 'statistik', 'karakternya']

=== Metrik Agregat ===
Recall@5:    0.0000
Precision@5: 0.0000
F1@5:        0.0000


### 7b. Semantic Relevance Score (dengan Baseline Kontrol)

Mengukur seberapa dekat secara makna kandidat yang dihasilkan sistem dengan jawaban resmi, **tanpa mensyaratkan exact match** -- sejalan dengan tujuan sistem sebagai penyedia kandidat pertimbangan, bukan penjawab tunggal. Disertai baseline kontrol (kemiripan terhadap kata tak-berhubungan) sebagai pembanding, supaya skor mentah tidak disalahartikan sebagai "persentase kemiripan" mutlak.

In [ ]:
def present_candidates(term, definition, topn_per_keyword=10):
    """
    Menghasilkan SELURUH kandidat dari FastText untuk satu istilah,
    disertai skor kemiripan makna (SBERT) sebagai informasi pendamping.
    Diurutkan menurun berdasarkan skor untuk kenyamanan baca, TANPA
    membuang/memotong kandidat manapun -- keputusan akhir diserahkan
    ke pengguna.
    """
    candidates, seeds = generate_candidates(term, definition, topn_per_keyword)

    scored_candidates = []
    for candidate in candidates:
        score = score_candidate(candidate, definition)
        scored_candidates.append((candidate, score))

    scored_candidates.sort(key=lambda x: x[1], reverse=True)
    return scored_candidates

In [ ]:
def semantic_relevance_score(term, definition, gold_answer, topn_per_keyword=10):
    """
    Mengukur kedekatan makna antara seluruh kandidat yang dihasilkan sistem
    dan jawaban gold standard, tanpa mensyaratkan exact match. Mengembalikan
    skor kemiripan TERTINGGI di antara kandidat -- merepresentasikan
    'kandidat paling mendekati makna gold standard yang berhasil dihasilkan
    sistem', berapapun urutannya.
    """
    results = present_candidates(term, definition, topn_per_keyword)
    emb_gold = get_sbert_embedding(gold_answer)

    best_score = -1.0
    best_word = None
    for word, _ in results:
        emb_word = get_sbert_embedding(word)
        sim = cosine_sim(emb_word, emb_gold)
        if sim > best_score:
            best_score = sim
            best_word = word

    return {
        "term": term,
        "gold_answer": gold_answer,
        "closest_candidate": best_word,
        "semantic_score": best_score,
    }


def evaluate_semantic_relevance(definitions_dict, gold_dict, topn_per_keyword=10):
    results = [
        semantic_relevance_score(term, definitions_dict[term], gold_dict[term], topn_per_keyword)
        for term in gold_dict
    ]
    avg_score = np.mean([r["semantic_score"] for r in results])
    return {"individual_results": results, "average_semantic_score": avg_score}

In [ ]:
sem_results = evaluate_semantic_relevance(definitions, gold_standard)

print("=== Semantic Relevance Score per Istilah ===")
for r in sem_results["individual_results"]:
    print(f"{r['term']:12s} -> gold: '{r['gold_answer']:15s}' | kandidat terdekat: '{r['closest_candidate']}' (skor: {r['semantic_score']:.4f})")

print()
print(f"=== Rata-rata Semantic Relevance Score: {sem_results['average_semantic_score']:.4f} ===")

=== Semantic Relevance Score per Istilah ===
array        -> gold: 'larik          ' | kandidat terdekat: 'acak' (skor: 0.9146)
browser      -> gold: 'peramban       ' | kandidat terdekat: 'kampung' (skor: 0.6206)
compiler     -> gold: 'pengompilasi   ' | kandidat terdekat: 'penggiling' (skor: 0.8833)
mouse        -> gold: 'tetikus        ' | kandidat terdekat: 'peranti' (skor: 0.8073)
string       -> gold: 'untai          ' | kandidat terdekat: 'mendata' (skor: 0.8831)

=== Rata-rata Semantic Relevance Score: 0.8218 ===


In [ ]:
# Baseline kontrol: skor kemiripan antara gold answer dan kata yang JELAS
# tidak berhubungan, untuk memberi titik pembanding objektif terhadap skor
# Semantic Relevance Score di atas (0.82). Tanpa baseline ini, skor mentah
# SBERT tidak punya makna mutlak (lihat diskusi keterbatasan pada Bagian 2).
unrelated_words = ["kucing", "matahari", "kursi", "hujan", "pisang"]

print("=== Baseline: kemiripan gold answer vs kata tidak berhubungan ===")
for term, gold in gold_standard.items():
    emb_gold = get_sbert_embedding(gold)
    scores = [cosine_sim(emb_gold, get_sbert_embedding(w)) for w in unrelated_words]
    print(f"{gold:15s} vs kata acak -> rata-rata skor: {np.mean(scores):.4f} "
          f"(min: {min(scores):.4f}, max: {max(scores):.4f})")

## Lampiran — Eksplorasi Awal (Tidak Dipakai di Pipeline Final)

Sel-sel di bawah ini **bukan bagian dari pipeline aktif**. Disimpan sebagai jejak proses iterasi riset: menunjukkan pendekatan yang dicoba lebih dulu, ditemukan keterbatasannya, lalu ditinggalkan demi pendekatan yang lebih baik di bagian utama notebook. Tidak perlu dijalankan ulang.

### A. Percobaan IndoBERT base (mean pooling)

Ditinggalkan karena masalah **anisotropy**: vektor dari mean pooling model BERT dasar (bukan yang dilatih khusus sentence-similarity) tidak konsisten merepresentasikan kemiripan makna -- misalnya "unduh" justru lebih mirip "kucing" daripada "ambil berkas dari internet". Lihat Reimers & Gurevych (2019) untuk penjelasan ini dalam literatur.

In [ ]:
# (Tidak dijalankan -- memerlukan instalasi AutoModel/AutoTokenizer IndoBERT
# yang sudah tidak dipakai di pipeline final)
#
# from transformers import AutoTokenizer, AutoModel
#
# INDOBERT_MODEL_NAME = "indobenchmark/indobert-base-p1"
# indobert_tokeniser = AutoTokenizer.from_pretrained(INDOBERT_MODEL_NAME)
# indobert_model = AutoModel.from_pretrained(INDOBERT_MODEL_NAME)
# indobert_model = indobert_model.to(device)
# indobert_model.eval()
#
# def get_bert_embedding(text):
#     inputs = indobert_tokeniser(text, return_tensors="pt", truncation=True, padding=True).to(device)
#     with torch.no_grad():
#         outputs = indobert_model(**inputs)
#     token_embeddings = outputs.last_hidden_state.squeeze(0)
#     sentence_embedding = token_embeddings.mean(dim=0)
#     return sentence_embedding.cpu().numpy()
#
# emb_unduh = get_bert_embedding("unduh")
# emb_download = get_bert_embedding("ambil berkas dari internet")
# emb_kucing = get_bert_embedding("kucing")
# print(cosine_sim(emb_unduh, emb_download))   # -> 0.4855
# print(cosine_sim(emb_unduh, emb_kucing))     # -> 0.6731 (seharusnya lebih kecil)

### B. Percobaan scoring v1: rata-rata kemiripan ke kata kunci individual

`score_candidate` versi pertama membandingkan kandidat dengan **setiap kata kunci secara terpisah** (kata vs kata), bukan dengan definisi penuh. Hasilnya cukup baik untuk membedakan kandidat relevan vs tidak relevan pada uji kata tunggal ("untai"/"string" vs "kucing"), tetapi pada pipeline penuh ditemukan bias sistematis: kandidat yang **mengandung secara harfiah** salah satu kata kunci (mis. "berkarakter" mengandung "karakter") selalu menang dibanding kandidat yang secara konsep tepat tapi tidak mirip secara leksikal (mis. "untai"). Beralih ke definisi-penuh-vs-kandidat (Bagian 2) mengurangi tapi tidak menghilangkan bias ini -- didiskusikan sebagai temuan analisis error di laporan.

In [ ]:
# def score_candidate_v1(candidate_word, keywords):
#     emb_candidate = get_sbert_embedding(candidate_word)
#     similarities = []
#     for kw in keywords:
#         emb_kw = get_sbert_embedding(kw)
#         similarities.append(cosine_sim(emb_candidate, emb_kw))
#     return np.mean(similarities)
#
# keywords_test = ["sequence", "characters", "data"]
# print(score_candidate_v1("untai", keywords_test))   # 0.4642
# print(score_candidate_v1("string", keywords_test))  # 0.4551
# print(score_candidate_v1("kucing", keywords_test))  # 0.2009 (arah benar di sini)

### C. Catatan: sirkularitas evaluasi akibat `LITERAL_TRANSLATION`

Versi awal pipeline memiliki `LITERAL_TRANSLATION` -- kamus terjemahan harfiah istilah utama (mis. "string" -> "untai") -- yang ditambahkan langsung sebagai kandidat tanpa melalui pencarian FastText. Setelah diperiksa, isi kamus ini **identik persis** dengan `gold_standard` untuk 4 dari 5 istilah uji. Akibatnya, gold answer selalu otomatis masuk ke himpunan kandidat, membuat seluruh metrik evaluasi (Coverage, Semantic Relevance Score) bernilai sempurna (1.0000) secara sirkular -- bukan karena kemampuan riil sistem menemukan jawaban tersebut.

`LITERAL_TRANSLATION` telah **dihapus sepenuhnya** dari pipeline final (lihat `generate_candidates` di Bagian 5). Sumber kandidat saat ini murni berasal dari kata kunci definisi (Jalur 1), tanpa jalur tambahan apapun yang bisa memasukkan jawaban secara langsung. Ini memastikan seluruh metrik evaluasi pada Bagian 7 mengukur kemampuan riil sistem, bukan artefak desain.

In [ ]:
# Sel ini sengaja dikosongkan dari kode aktif -- sebagai pengingat,
# berikut bentuk LITERAL_TRANSLATION yang TELAH DIHAPUS dari pipeline,
# disimpan hanya sebagai dokumentasi historis:
#
# LITERAL_TRANSLATION = {
#     "string": "untai", "array": "larik", "browser": "peramban",
#     "compiler": "pengompilasi", "mouse": "tetikus",
# }
#
# Perhatikan kamus ini identik dengan gold_standard -- itulah sumber
# sirkularitasnya.